In [1]:
# import numpy as np
# import matplotlib.pyplot as plt

# def compute_rollout_attention(att_mat: np.ndarray) -> np.ndarray:
#     """
#     Compute the rollout attention across layers.
    
#     att_mat: shape = (n_layers, n_tokens, n_tokens)
#              Typically n_tokens = 1 + # of patch tokens.
    
#     Returns:
#     - final_rollout: shape = (n_tokens, n_tokens)
#                      The final aggregated attention after multiplying all layers.
#     """
#     n_layers, n_tokens, _ = att_mat.shape
    
#     # Step 1: add identity to each layer's attention (so each token also attends to itself)
#     # We broadcast-add the identity along the first dimension (layers).
#     att_mat_res = att_mat + np.eye(n_tokens)[None, :, :]
    
#     # Step 2: normalize each row so it sums to 1
#     # axis=-1 => row-wise normalization
#     att_mat_res = att_mat_res / att_mat_res.sum(axis=-1, keepdims=True)

#     # Step 3: cumulative multiplication of the layers
#     # Start from the first layer, then keep multiplying the next layer
#     #  (Think of it like matrix chain multiplication.)
#     rollout = att_mat_res[0]  # shape (n_tokens, n_tokens)
#     for i in range(1, n_layers):
#         rollout = att_mat_res[i].dot(rollout)
    
#     return rollout

# att_path = "../output/cliphba_meg_viz/test_images/attention_weights.npy"
# att = np.load(att_path)
# print(att.shape) # shape (2, 24, 257, 257) # (batch_size, n_layers, n_tokens, n_tokens) # first token is [CLS] token

# weighting_matrix_path = "../output/cliphba_meg_viz/test_images/weighting_matrix.npy"
# weighting_matrix = np.load(weighting_matrix_path)
# print("weighting_matrix.shape:", weighting_matrix.shape) # shape = (281, 24) # (n_timepoints, n_layers)

# visual_scaler_path = "../output/cliphba_meg_viz/test_images/visual_scaler.npy"
# visual_scaler = np.load(visual_scaler_path)
# print("visual_scaler.shape:", visual_scaler.shape) # shape = (281, 1) # (n_timepoints, 1)

# # for each row in weighting_matrix, minmax to 0-1 and make sure it sumes up to 1
# weighting_matrix = weighting_matrix - np.min(weighting_matrix, axis=0)
# weighting_matrix = weighting_matrix / np.sum(weighting_matrix, axis=0)


# plt.imshow(weighting_matrix, aspect='auto', cmap='viridis')
# plt.colorbar()
# plt.show()


# RICE

In [2]:
# import os
# import shutil
# import warnings
# warnings.filterwarnings("ignore", category=UserWarning)

# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# from PIL import Image
# from tqdm import tqdm

# import torch
# from torch.utils.data import DataLoader, Dataset
# from torchvision import transforms
# import torch.nn.functional as F

# # -------------------------
# # 1. IMPORT YOUR MODEL CODE
# # -------------------------
# # The following import assumes you have these in 'train_dynamic_success.py'.
# # If your file name or function names differ, adjust accordingly.
# from train_dynamic_success import (
#     seed_everything,  # Function to fix random seeds
#     CLIPHBA,          # Your CLIP-HBA dynamic class
#     apply_dora_to_ViT,
#     apply_lora_to_ViT,
#     classnames66
# )


# # -------------------------
# # 2. DATASET DEFINITION
# # -------------------------
# class ImageDataset(Dataset):
#     """
#     Loads images from a directory or a single image file, applying a
#     predefined transform (224x224, normalization, etc.).
#     """
#     def __init__(self, img_path):
#         """
#         img_path can be either a directory containing images 
#         or a path to a single image.
#         """
#         if os.path.isdir(img_path):
#             self.img_dir = img_path
#             self.image_names = [
#                 img for img in sorted(os.listdir(img_path))
#                 if img.lower().endswith(('.png', '.jpg', '.jpeg'))
#             ]
#         elif os.path.isfile(img_path) and img_path.lower().endswith(('.png', '.jpg', '.jpeg')):
#             self.img_dir, single_image_name = os.path.split(img_path)
#             self.image_names = [single_image_name]  # Single image in a list
#         else:
#             raise ValueError(
#                 f"Provided path '{img_path}' is neither a directory nor a file, "
#                 "or the file type is not supported."
#             )

#         self.transform = transforms.Compose([
#             transforms.Resize((224, 224)),
#             transforms.ToTensor(),
#             transforms.Normalize(
#                 mean=[0.52997664, 0.48070561, 0.41943838],
#                 std=[0.27608301, 0.26593025, 0.28238822]
#             )
#         ])

#     def __len__(self):
#         return len(self.image_names)

#     def __getitem__(self, index):
#         image_name = self.image_names[index]
#         img_path = os.path.join(self.img_dir, image_name)
#         image = Image.open(img_path).convert('RGB')  # Ensure image is RGB
#         image = self.transform(image)
#         return image_name, image


# # -------------------------
# # 3. OPTIONAL RDM FUNCTION
# # -------------------------
# def compute_temporal_object_rdm(emb):
#     """
#     Example function from your snippet if you need RDMs.
#     """
#     num_timepoints, num_objects, _ = emb.shape
#     rdm_array = np.zeros((num_timepoints, num_objects, num_objects))
#     for t in range(num_timepoints):
#         # Compute the Pearson correlation matrix for objects at timepoint t
#         corr_matrix = np.corrcoef(emb[t])
#         # Convert correlation to Pearson distance (1 - correlation)
#         rdm_array[t] = 1 - corr_matrix
#     return rdm_array


# # -------------------------
# # 4. MODEL INFERENCE HELPER
# # -------------------------
# def run_image(model, dataset, data_loader, save_folder, embedding_save_folder,
#               inference_start, inference_step, inference_end, device=torch.device("cuda"), n_dim=66):
#     """
#     Original function for running inference on an entire dataset folder.
#     Used here if you want to generate embeddings for all images.
#     """
#     model.eval()
#     model.to(device)

#     n_images = len(dataset)
#     with torch.no_grad():
#         n_inference_timepoints = len(range(inference_start, inference_end + 1, inference_step))
#         image_names = []
#         output_embs = np.zeros((n_inference_timepoints, n_images, n_dim))

#         current_idx = 0
#         progress_bar = tqdm(enumerate(data_loader), total=len(data_loader), desc=f"Processing images")
#         for batch_idx, (batch_image_names, batch_images) in progress_bar:
#             batch_images = batch_images.to(device)
#             image_names.extend(batch_image_names)

#             # Forward pass (pred_embs shape: (n_timepoints, batch_size, n_dim))
#             pred_embs, _, _ = model(batch_images)

#             # Move to CPU
#             pred_embs = pred_embs.cpu().numpy()
#             batch_size = batch_images.size(0)

#             # Store
#             output_embs[:, current_idx:current_idx + batch_size, :] = pred_embs
#             current_idx += batch_size

#         print(f"Finished processing {n_images} images")
#         print(f"output_embs shape: {output_embs.shape}")

#         # Example normalization (optional)
#         output_embs = (output_embs - np.min(output_embs)) / (np.max(output_embs) - np.min(output_embs)) * 100

#         # Save CSV for each timepoint
#         for i in range(n_inference_timepoints):
#             timepoint = inference_start + i * inference_step
#             hba_embedding = pd.DataFrame(output_embs[i])
#             hba_embedding['image'] = image_names
#             hba_embedding = hba_embedding[['image'] + [col for col in hba_embedding if col != 'image']]
#             emb_save_path = f"{embedding_save_folder}/dynamic_embedding_{timepoint}ms.csv"
#             hba_embedding.to_csv(emb_save_path, index=False)
#             print(f"Embedding saved to {emb_save_path}")

#         # (Optional) Compute RDM
#         output_rdms = compute_temporal_object_rdm(output_embs)

#         np.save(f"{save_folder}/embeddings_{inference_start}ms-{inference_end}ms-{inference_step}step.npy", output_embs)
#         np.save(f"{save_folder}/rdms_{inference_start}ms-{inference_end}ms-{inference_step}step.npy", output_rdms)
#     return output_embs, output_rdms


# # -------------------------
# # 5. SALIENCY SUPPORT
# # -------------------------
# def generate_masks(num_masks, mask_size, p1, input_size=(224, 224)):
#     """
#     Generate random binary masks of shape (num_masks, H, W, 1).

#     :param num_masks: number of masks to generate
#     :param mask_size: defines the coarse grid size s x s
#     :param p1: probability of a cell being 1
#     :param input_size: the final size (H, W) we need for each mask
#     """
#     import math
#     from skimage.transform import resize

#     H, W = input_size
#     # Each grid cell covers (H/s, W/s) in the upsample
#     cell_size = np.ceil(np.array([H, W]) / mask_size).astype(int)
#     up_size = (mask_size + 1) * cell_size  # bigger so we can random shift

#     # Create the coarse random grid for all masks
#     grid = np.random.rand(num_masks, mask_size, mask_size) < p1
#     grid = grid.astype('float32')

#     # Placeholder for final masks
#     masks = np.zeros((num_masks, H, W), dtype=np.float32)

#     for i in tqdm(range(num_masks), desc='Generating masks'):
#         # Random shift
#         shift_x = np.random.randint(0, cell_size[0])
#         shift_y = np.random.randint(0, cell_size[1])

#         # Upsample the s x s grid to up_size, then crop to input_size
#         upsampled = resize(grid[i], up_size, order=1, mode='reflect', anti_aliasing=False)
#         crop = upsampled[shift_x:shift_x + H, shift_y:shift_y + W]
#         masks[i] = crop

#     # Add a channel dimension: (num_masks, H, W, 1)
#     masks = np.expand_dims(masks, axis=-1)
#     return masks


# def compute_saliency(model, device, original_image, anchor_emb, masks):
#     """
#     Compute a simple saliency map by evaluating how each masked image's
#     embedding changes in similarity compared to the anchor embedding.

#     :param model: your CLIPHBA dynamic model
#     :param device: torch device
#     :param original_image: shape (1, 3, 224, 224), the unmasked input
#     :param anchor_emb: the model's embedding for the unmasked image, shape (n_timepoints, n_dim)
#     :param masks: shape (N, 224, 224, 1) of binary masks
#     :return: saliency map of shape (224, 224)
#     """
#     model.eval()

#     # Convert original_image to CPU numpy for overlay, but we'll keep it in Torch for forward pass
#     # We assume original_image is batch_size=1
#     batch_size = masks.shape[0]
#     # Prepare masked images
#     # Masks should broadcast from (N, 224, 224, 1) -> (N, 3, 224, 224)
#     # However, the model expects (N, 3, 224, 224). We'll tile the mask across the 3 channels.
#     masks_3ch = np.repeat(masks, 3, axis=-1)  # (N, 224, 224, 3)
#     masks_3ch = np.transpose(masks_3ch, (0, 3, 1, 2))  # => (N, 3, 224, 224)

#     # Torch conversion
#     masks_torch = torch.from_numpy(masks_3ch).float().to(device)
#     original_image_torch = original_image.to(device)

#     # Expand original_image from (1, 3, 224, 224) -> (N, 3, 224, 224) by repeating
#     original_image_batched = original_image_torch.repeat(batch_size, 1, 1, 1)

#     # Apply the masks
#     masked_images = original_image_batched * masks_torch

#     # Forward pass with masked images
#     with torch.no_grad():
#         pred_embs, _, _ = model(masked_images)  # shape: (n_timepoints, N, n_dim)
#         # Move to CPU
#         pred_embs = pred_embs.cpu().numpy()

#     # anchor_emb shape: (n_timepoints, n_dim)
#     # pred_embs shape:  (n_timepoints, N, n_dim)

#     # Compute similarity relative to anchor_emb
#     # We'll do average across timepoints or pick a specific timepoint, etc.
#     # Let's do an average across timepoints to get a single similarity scalar for each mask.
#     n_timepoints = pred_embs.shape[0]
#     # anchor_emb => (1, n_timepoints, n_dim) so we can broadcast
#     anchor_emb_expanded = np.expand_dims(anchor_emb, axis=1)  # => (n_timepoints, 1, n_dim)

#     # Cosine similarity for each timepoint, each mask
#     # cos_sim(t, i) = anchor_emb[t] dot pred_embs[t, i] / (||anchor|| * ||pred_emb||)
#     # We'll average across timepoints afterwards.
#     similarities = []
#     for t in range(n_timepoints):
#         a = anchor_emb_expanded[t]  # shape (1, n_dim)
#         b = pred_embs[t]            # shape (N, n_dim)
#         # Normalize
#         a_norm = a / (np.linalg.norm(a, axis=-1, keepdims=True) + 1e-9)
#         b_norm = b / (np.linalg.norm(b, axis=-1, keepdims=True) + 1e-9)
#         # Dot product
#         sim_t = np.sum(a_norm * b_norm, axis=-1)  # shape (N,)
#         similarities.append(sim_t)
#     # shape => (n_timepoints, N)
#     similarities = np.array(similarities)
#     # Average across timepoints => shape (N,)
#     avg_similarities = np.mean(similarities, axis=0)

#     # The saliency at each mask is how *much* it differs from the anchor
#     # We'll define it as (1 - avg_sim) so smaller similarity => bigger difference => bigger saliency
#     diffs = 1.0 - avg_similarities  # shape (N,)

#     # Now we want a per-pixel saliency. A common approach is to "add up" these differences
#     # weighted by whether a pixel was "masked" or "not masked".
#     # However, here, each mask is 1 or 0 in each pixel. If a pixel is 0 in many masks that produce
#     # big differences, that pixel might be more important.
#     # We'll do a very simple approach: sum up the diff for each mask where that pixel is 0.

#     # Convert masks => (N, 224, 224)
#     masks_2d = masks[..., 0]  # shape => (N, 224, 224)
#     sal_map = np.zeros((masks.shape[1], masks.shape[2]), dtype=np.float32)

#     # For each pixel, we gather those mask indices that covered the pixel (0 => covered, 1 => kept)
#     # Actually, to measure "importance," we might want to see how similarity changes if we zero out that pixel
#     # So we look for mask == 0 in that pixel. Then we sum the diffs for those masks.
#     # This is a naive approach. More advanced methods do a weighted average, etc.
#     for i in range(masks.shape[1]):      # 224
#         for j in range(masks.shape[2]):  # 224
#             # mask pixel == 0 => covered
#             covered_mask_ids = np.where(masks_2d[:, i, j] < 0.5)[0]
#             if len(covered_mask_ids) > 0:
#                 sal_map[i, j] = np.mean(diffs[covered_mask_ids])
#             else:
#                 sal_map[i, j] = 0.0

#     # Optionally normalize sal_map to [0,1]
#     sal_map -= sal_map.min()
#     if sal_map.max() > 0:
#         sal_map /= sal_map.max()

#     return sal_map


# def overlay_saliency_on_image(image_np, saliency_map, alpha=0.6, cmap='jet'):
#     """
#     Overlay a heatmap on top of an original image.
#     :param image_np: (H, W, 3) in [0,1] or [0,255]
#     :param saliency_map: (H, W) in [0,1]
#     :param alpha: blending factor
#     :param cmap: matplotlib colormap
#     :return: overlayed image as a numpy array
#     """
#     import matplotlib.cm as cm
#     # Ensure image is float in [0,1]
#     if image_np.dtype != np.float32 and image_np.dtype != np.float64:
#         image_np = image_np.astype(np.float32)
#     if image_np.max() > 1.0:
#         image_np /= 255.0

#     # Convert saliency to color
#     colormap = cm.get_cmap(cmap)
#     saliency_color = colormap(saliency_map)[:, :, :3]  # shape (H, W, 3)

#     # Blend
#     overlay = (1 - alpha) * image_np + alpha * saliency_color
#     overlay = np.clip(overlay, 0, 1)
#     return overlay

# def denormalize(image_tensor, mean, std):
#     """
#     Denormalize an image tensor from [-mean/std] normalization back to [0, 1] or [0, 255].
#     Args:
#         image_tensor: Tensor of shape (3, H, W) or (1, 3, H, W)
#         mean: List of means for each channel (RGB).
#         std: List of standard deviations for each channel (RGB).
#     Returns:
#         Denormalized image as a numpy array of shape (H, W, 3).
#     """
#     # If the input is batched (1, 3, H, W), remove the batch dimension
#     if len(image_tensor.shape) == 4:
#         image_tensor = image_tensor.squeeze(0)
#     # Denormalize
#     mean = np.array(mean).reshape(3, 1, 1)
#     std = np.array(std).reshape(3, 1, 1)
#     image_tensor = image_tensor.cpu().numpy() * std + mean  # Undo normalization
#     # Transpose to (H, W, 3) and clip to valid range [0, 1]
#     image_tensor = np.clip(np.transpose(image_tensor, (1, 2, 0)), 0, 1)
#     return image_tensor


# # -------------------------
# # 6. MAIN SCRIPT
# # -------------------------
# if __name__ == '__main__':
#     seed_everything(1)

#     ########################################
#     # User-defined paths and parameters
#     ########################################
#     img_dir = './Data/test_images/'
#     n_dim = 66  # options: 49, 66
#     backbone = 'ViT-L/14'
#     model_path = './models/cliphba_dynamic_official_v9_step.pth'
#     save_folder = '../output/cliphba_dynamic_66d_official_v2/viz/'
#     batch_size = 128
#     vision_layers = 24
#     transformer_layers = 1
#     rank = 32
#     ms_start, ms_end, ms_step = -100, 1300, 5  # model training range
#     inference_start, inference_end, inference_step, train_window_size = -100, 1300, 5, 15
#     cuda = 'cuda:0'
#     load_pretrained = True
#     ########################################

#     # Reinitialize the save_folder
#     if os.path.exists(save_folder):
#         shutil.rmtree(save_folder)
#         print(f"Reinitialized save folder: {save_folder}")
#     os.makedirs(save_folder, exist_ok=True)

#     # Create embedding folder
#     embedding_save_folder = f"{save_folder}/emb"
#     print(f"Embedding will be saved to folder: {embedding_save_folder}")
#     if os.path.exists(embedding_save_folder):
#         shutil.rmtree(embedding_save_folder)
#     os.makedirs(embedding_save_folder, exist_ok=True)

#     # Get classnames
#     if n_dim == 66:
#         classnames = classnames66
#     else:
#         raise ValueError("n_dim must be 66")

#     classnames = [x[0] for x in classnames]

#     # Setup positional embeddings
#     if backbone == 'RN50':
#         pos_embedding = False
#     elif backbone in ['ViT-B/16', 'ViT-B/32', 'ViT-L/14']:
#         pos_embedding = True
#     else:
#         raise ValueError(f"Unknown backbone {backbone}")

#     # Create weighting matrix
#     weighting_matrix = None

#     # -------------------------
#     # Initialize Model
#     # -------------------------
#     model = CLIPHBA(
#         classnames=classnames,
#         weighting_matrix=weighting_matrix,
#         backbone_name=backbone,
#         pos_embedding=pos_embedding,
#         ms_start=ms_start,
#         ms_step=ms_step,
#         ms_end=ms_end,
#         train_start=inference_start,
#         train_step=inference_step,
#         train_end=inference_end,
#         train_window_size=train_window_size
#     )

#     # Load pretrained
#     if load_pretrained:
#         apply_dora_to_ViT(model, n_vision_layers=vision_layers, n_transformer_layers=transformer_layers, r=rank, dora_dropout=0.1)
#         print(f'Loading Pretrained Model: "{model_path}"')
#         model_state_dict = torch.load(model_path, map_location='cpu')
#         adjusted_state_dict = {key.replace("module.", ""): value for key, value in model_state_dict.items()}
#         model.load_state_dict(adjusted_state_dict)
#         print('Model Loaded Successfully\n')
#     else:
#         raise ValueError("No model loaded")
    
#     for name, param in model.named_parameters():
#         if torch.isnan(param).any():
#             print("NaN in param:", name)
#             raise ValueError("NaN in param")
    

#     device = torch.device(cuda)

#     # Save some model state if needed
#     np.save(f"{save_folder}/weighting_matrix.npy", np.array(model.clip_model.weighting_matrix))

#     # -------------------------
#     # Load Data
#     # -------------------------
#     dataset = ImageDataset(img_dir)
#     data_loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

#     # If you want to run for all images:
#     # run_image(model, dataset, data_loader, save_folder, embedding_save_folder,
#     #           inference_start, inference_step, inference_end, device=device, n_dim=n_dim)

#     # -------------------------
#     # SALIENCY ON THE FIRST IMAGE ONLY
#     # -------------------------
#     # 1) Grab the first image from the dataset
#     image_name, first_image = dataset[0]  # shape => (3, 224, 224) in Torch
#     first_image = first_image.unsqueeze(0)  # shape => (1, 3, 224, 224)
#     print(f"\nWorking with the first image: {image_name}")

#     # 2) Get the anchor embedding for the unmasked image
#     model.eval()
#     model.to(device)
#     with torch.no_grad():
#         first_image_device = first_image.to(device)
#         anchor_emb, _, _ = model(first_image_device)  # shape => (n_timepoints, 1, n_dim)
#         anchor_emb = anchor_emb.squeeze(1).cpu().numpy()  # shape => (n_timepoints, n_dim)

#     print(f"Anchor embedding shape: {anchor_emb.shape}")

#     # 3) Generate random masks
#     # Number of masks can vary. Larger => better coverage but more compute.
#     # Probability p1 => fraction of '1' in the coarse grid.
#     # mask_size => e.g., 8 or 16. 
#     num_masks = 500
#     s = 8
#     p1 = 0.5
#     masks = generate_masks(num_masks, s, p1, input_size=(224, 224))  # shape => (N, 224, 224, 1)

#     # 4) Compute saliency map
#     sal_map = compute_saliency(
#         model=model,
#         device=device,
#         original_image=first_image,  # (1,3,224,224)
#         anchor_emb=anchor_emb,       # (n_timepoints, n_dim)
#         masks=masks                  # (N,224,224,1)
#     )

#     # 5) Overlay saliency on the original image
#     # Convert the first image to a numpy array in [0,255] for visualization
#     # Denormalize the image
#     mean = [0.52997664, 0.48070561, 0.41943838]
#     std = [0.27608301, 0.26593025, 0.28238822]
#     unnorm_img = denormalize(first_image, mean, std)  # shape => (224, 224, 3)


#     overlay = overlay_saliency_on_image(unnorm_img, sal_map, alpha=0.5, cmap='jet')

#     # 6) Plot and save
#     fig, ax = plt.subplots(1, 2, figsize=(8, 4))
#     ax[0].imshow(unnorm_img)
#     ax[0].set_title("Original Image")
#     ax[0].axis("off")

#     ax[1].imshow(overlay)
#     ax[1].set_title("Saliency Overlay")
#     ax[1].axis("off")

#     plt.tight_layout()
#     plt.show()

#     # Optionally save the saliency overlay
#     out_path = os.path.join(save_folder, f"{image_name}_saliency.png")
#     plt.imsave(out_path, overlay)
#     print(f"Saliency map saved to {out_path}")


In [2]:
import inspect
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import os
import shutil
from PIL import Image
import numpy as np
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import torch
from torch.nn import DataParallel
from sklearn.metrics.pairwise import cosine_similarity
from scipy.spatial.distance import pdist, squareform
import h5py
import warnings
import imageio
warnings.filterwarnings("ignore", category=UserWarning)

# -------------------------
# 1. IMPORT YOUR MODEL CODE
# -------------------------
from train_dynamic_step import (
    seed_everything,
    CLIPHBA,
    apply_dora_to_ViT,
    classnames66
)

# -------------------------
# 2. DATASET DEFINITION
# -------------------------
class ImageDataset(Dataset):
    def __init__(self, img_path):
        if os.path.isdir(img_path):
            self.img_dir = img_path
            self.image_names = [
                img for img in sorted(os.listdir(img_path))
                if img.lower().endswith(('.png', '.jpg', '.jpeg'))
            ]
        elif os.path.isfile(img_path) and img_path.lower().endswith(('.png', '.jpg', '.jpeg')):
            self.img_dir, single_image_name = os.path.split(img_path)
            self.image_names = [single_image_name]
        else:
            raise ValueError(
                f"Provided path '{img_path}' is neither a directory nor a file, "
                "or the file type is not supported."
            )

        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.52997664, 0.48070561, 0.41943838],
                std=[0.27608301, 0.26593025, 0.28238822]
            )
        ])

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, index):
        image_name = self.image_names[index]
        img_path = os.path.join(self.img_dir, image_name)
        image = Image.open(img_path).convert('RGB')
        image = self.transform(image)
        return image_name, image


# -------------------------
# 3. OPTIONAL RDM FUNCTION
# -------------------------
def compute_temporal_object_rdm(emb):
    """
    Example function if you need RDMs.
    emb shape: (num_timepoints, num_objects, embedding_dim)
    """
    num_timepoints, num_objects, _ = emb.shape
    rdm_array = np.zeros((num_timepoints, num_objects, num_objects))
    for t in range(num_timepoints):
        corr_matrix = np.corrcoef(emb[t])
        rdm_array[t] = 1 - corr_matrix
    return rdm_array


# -------------------------
# 4. SALIENCY SUPPORT
# -------------------------
def generate_masks(num_masks, mask_size, p1, input_size=(224, 224)):
    """
    Generate random binary masks of shape (num_masks, H, W, 1).
    """
    from skimage.transform import resize

    H, W = input_size
    cell_size = np.ceil(np.array([H, W]) / mask_size).astype(int)
    up_size = (mask_size + 1) * cell_size
    grid = np.random.rand(num_masks, mask_size, mask_size) < p1
    grid = grid.astype('float32')
    masks = np.zeros((num_masks, H, W), dtype=np.float32)

    for i in tqdm(range(num_masks), desc='Generating masks'):
        shift_x = np.random.randint(0, cell_size[0])
        shift_y = np.random.randint(0, cell_size[1])
        upsampled = resize(
            grid[i],
            up_size,
            order=1,
            mode='reflect',
            anti_aliasing=False
        )
        crop = upsampled[shift_x:shift_x + H, shift_y:shift_y + W]
        masks[i] = crop

    masks = np.expand_dims(masks, axis=-1)
    return masks


def compute_saliency_all_timepoints(model, device, original_image, anchor_emb, masks):
    """
    Computes a separate saliency map for EACH timepoint.
    anchor_emb: (n_timepoints, embedding_dim)
    pred_embs:  (n_timepoints, N, embedding_dim)
    """
    model.eval()
    model.to(device)

    batch_size = masks.shape[0]
    H, W = masks.shape[1], masks.shape[2]

    # Expand masks to 3 channels
    masks_3ch = np.repeat(masks, 3, axis=-1)
    masks_3ch = np.transpose(masks_3ch, (0, 3, 1, 2))

    masks_torch = torch.from_numpy(masks_3ch).float().to(device)
    original_image_batched = original_image.to(device).repeat(batch_size, 1, 1, 1)
    masked_images = original_image_batched * masks_torch

    with torch.no_grad():
        # pred_embs => shape (n_timepoints, N, embedding_dim)
        pred_embs, _, _ = model(masked_images)
        pred_embs = pred_embs.cpu().numpy()
        # print(pred_embs)

    n_timepoints = pred_embs.shape[0]
    anchor_norm = np.linalg.norm(anchor_emb, axis=-1, keepdims=True) + 1e-9
    sal_map_all = np.zeros((n_timepoints, H, W), dtype=np.float32)
    masks_2d = masks[..., 0]

    for t_idx in tqdm(range(n_timepoints), desc='Computing saliency for all timepoints'):
        a = anchor_emb[t_idx]
        a_norm = anchor_norm[t_idx]
        a_normed = a / a_norm

        b = pred_embs[t_idx]
        b_norm = b / (np.linalg.norm(b, axis=-1, keepdims=True) + 1e-9)

        cos_sim = np.sum(a_normed * b_norm, axis=-1)
        diffs = 1.0 - cos_sim
        sal_map_t = np.zeros((H, W), dtype=np.float32)

        for i in range(H):
            for j in range(W):
                covered_mask_ids = np.where(masks_2d[:, i, j] < 0.5)[0]
                if len(covered_mask_ids) > 0:
                    sal_map_t[i, j] = np.mean(diffs[covered_mask_ids])

        # Normalize [0,1]
        sal_map_t -= sal_map_t.min()
        if sal_map_t.max() > 0:
            sal_map_t /= sal_map_t.max()

        sal_map_all[t_idx] = sal_map_t

    return sal_map_all


def overlay_saliency_on_image(image_np, saliency_map, alpha=0.6, cmap='jet'):
    import matplotlib.cm as cm
    if image_np.dtype not in [np.float32, np.float64]:
        image_np = image_np.astype(np.float32)
    if image_np.max() > 1.0:
        image_np /= 255.0

    colormap = cm.get_cmap(cmap)
    saliency_color = colormap(saliency_map)[:, :, :3]
    overlay = (1 - alpha) * image_np + alpha * saliency_color
    overlay = np.clip(overlay, 0, 1)
    return overlay


def denormalize(image_tensor, mean, std):
    if len(image_tensor.shape) == 4:
        image_tensor = image_tensor.squeeze(0)
    mean = np.array(mean).reshape(3, 1, 1)
    std = np.array(std).reshape(3, 1, 1)
    image_tensor = image_tensor.cpu().numpy() * std + mean
    image_tensor = np.transpose(image_tensor, (1, 2, 0))
    image_tensor = np.clip(image_tensor, 0, 1)
    return image_tensor


def create_gif_from_saliency(
    unnorm_img,
    sal_map_all,
    timepoints,
    save_path="dynamic_saliency.gif",
    alpha=0.5,
    cmap='jet',
    fps=5
):
    frames = []
    for i, ms_val in tqdm(enumerate(timepoints), desc='Creating GIF frames'):
        overlay = overlay_saliency_on_image(unnorm_img, sal_map_all[i], alpha=alpha, cmap=cmap)
        fig, ax = plt.subplots(figsize=(6, 6))
        ax.imshow(overlay)
        ax.set_title(f"Time: {ms_val} ms", fontsize=14, color='red', pad=10)
        ax.axis('off')
        fig.canvas.draw()
        w, h = fig.canvas.get_width_height()
        buf = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
        buf = buf.reshape(h, w, 3)
        frames.append(buf)
        plt.close(fig)

    print(f"Saving GIF...")
    imageio.mimsave(save_path, frames, fps=fps)
    print(f"GIF saved to: {save_path}")


def check_nan_hook(module, inputs, output):
    """
    Forward hook that checks for NaNs in inputs and outputs (handles tuples).
    """
    # Outputs
    if isinstance(output, tuple):
        for i, out in enumerate(output):
            if torch.is_tensor(out) and torch.isnan(out).any():
                print(f"NaN in output[{i}] of module: {module.__class__.__name__}")
                raise RuntimeError("NaN encountered in module output.")
    else:
        if torch.is_tensor(output) and torch.isnan(output).any():
            print(f"NaN in output of module: {module.__class__.__name__}")
            raise RuntimeError("NaN encountered in module output.")

    # Inputs
    for i, inp in enumerate(inputs):
        if torch.is_tensor(inp) and torch.isnan(inp).any():
            print(f"NaN in input[{i}] of module: {module.__class__.__name__}")
            raise RuntimeError("NaN encountered in module input.")


# -------------------------
# 5. MAIN SCRIPT
# -------------------------
if __name__ == '__main__':
    seed_everything(1)

    # -------------------------
    # User-defined parameters
    # -------------------------
    img_dir = './Data/test_images/'
    n_dim = 66
    backbone = 'ViT-L/14'
    # model_path = './models/cliphba_dynamic_official_v9_step.pth'
    # save_folder = '../output/cliphba_dynamic_66d_official_v2/viz/'
    model_path = './models/test_models/cichy_individual_transfer/cliphba_dynamic_individual_cichy_p1.pth'
    save_folder = '../output/cliphba_individual_visualization/'
    batch_size = 128
    vision_layers = 24
    transformer_layers = 1
    rank = 32
    # ms_start, ms_end, ms_step = -100, 1300, 5
    ms_start, ms_end, ms_step = -100, 1000, 5
    cuda = 'cuda:0'
    load_pretrained = True

    # Reinitialize output folders
    if os.path.exists(save_folder):
        shutil.rmtree(save_folder)
        print(f"Reinitialized save folder: {save_folder}")
    os.makedirs(save_folder, exist_ok=True)
    emb_save_folder = os.path.join(save_folder, "emb")
    os.makedirs(emb_save_folder, exist_ok=True)

    if n_dim == 66:
        classnames = classnames66
    else:
        raise ValueError("n_dim must be 66")
    classnames = [x[0] for x in classnames]

    if backbone == 'RN50':
        pos_embedding = False
    elif backbone in ['ViT-B/16', 'ViT-B/32', 'ViT-L/14']:
        pos_embedding = True
    else:
        raise ValueError(f"Unknown backbone {backbone}")

    weighting_matrix = None

    # Instantiate Model
    model = CLIPHBA(
        classnames=classnames,
        weighting_matrix=weighting_matrix,
        backbone_name=backbone,
        pos_embedding=pos_embedding,
        ms_start=ms_start,
        ms_step=ms_step,
        ms_end=ms_end,
        train_start=ms_start,
        train_step=ms_step,
        train_end=ms_end,
        train_window_size=15
    )

    # Load pretrained weights
    if load_pretrained:
        apply_dora_to_ViT(model, n_vision_layers=vision_layers, n_transformer_layers=transformer_layers,
                          r=rank, dora_dropout=0.1)
        print(f'Loading Pretrained Model: "{model_path}"')
        model_state_dict = torch.load(model_path, map_location='cpu')
        adjusted_state_dict = {k.replace("module.", ""): v for k, v in model_state_dict.items()}
        model.load_state_dict(adjusted_state_dict)
        print('Model Loaded Successfully\n')
    else:
        raise ValueError("No pretrained model loaded!")

    device = torch.device(cuda)

    # Load the dataset (single image or directory)
    dataset = ImageDataset(img_dir)
    data_loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    # Grab the first image
    image_name, first_image = dataset[0]  # (3, 224, 224)
    first_image = first_image.unsqueeze(0)  # (1, 3, 224, 224)
    print("image:", first_image.shape)
    print(f"Working with the first image: {image_name}")

    model.eval()
    model.to(device)

    # Register NaN hooks
    for submodule in model.modules():
        submodule.register_forward_hook(check_nan_hook)

    # -------------------------
    # Avoid batch_size=1 trick
    # -------------------------
    first_image_device = first_image.to(device)
    # Create a dummy image (same shape). This ensures batch_size=2
    dummy_image = torch.zeros_like(first_image_device)
    combined_input = torch.cat([first_image_device, dummy_image], dim=0)  # shape (2, 3, 224, 224)

    # Forward pass, then index out the real image
    with torch.no_grad():
        anchor_emb_3d, _, _ = model(combined_input)  # shape => (n_timepoints, 2, n_dim)
        # Index out just the first image
        anchor_emb = anchor_emb_3d[:, 0, :].cpu().numpy()  # shape => (n_timepoints, n_dim)
        # print("Anchor embedding shape:", anchor_emb.shape)
        # print(anchor_emb)

    # 3) Generate random masks
    num_masks = 1000
    s = 8
    p1 = 0.5
    masks = generate_masks(num_masks, s, p1, input_size=(224, 224))

    # 4) Compute saliency for EVERY single timepoint
    sal_map_all = compute_saliency_all_timepoints(
        model=model,
        device=device,
        original_image=first_image,  # shape (1,3,224,224)
        anchor_emb=anchor_emb,       # shape (n_timepoints, n_dim)
        masks=masks
    )

    # 5) Denormalize the original image for visualization
    mean = [0.52997664, 0.48070561, 0.41943838]
    std = [0.27608301, 0.26593025, 0.28238822]
    unnorm_img = denormalize(first_image, mean, std)  # => (224,224,3) in [0,1]

    # 6) Create a GIF of all timepoints
    timepoints = list(range(ms_start, ms_end + 1, ms_step))  # e.g., -100, -95,...,1300
    image_name_no_ext = os.path.splitext(image_name)[0]
    gif_path = os.path.join(save_folder, f"dynamic_saliency_{image_name_no_ext}.gif")
    create_gif_from_saliency(
        unnorm_img=unnorm_img,
        sal_map_all=sal_map_all,
        timepoints=timepoints,
        save_path=gif_path,
        alpha=0.5,
        cmap='jet',
        fps=24
    )


Reinitialized save folder: ../output/cliphba_individual_visualization/
Loading Pretrained Model: "./models/test_models/cichy_individual_transfer_v2/cliphba_dynamic_individual_cichy_p3.pth"
Model Loaded Successfully

image: torch.Size([1, 3, 224, 224])
Working with the first image: aardvark_01b.jpg


Computing saliency for all timepoints: 100%|██████████| 221/221 [03:34<00:00,  1.03it/s]
Creating GIF frames: 0it [00:00, ?it/s]C:\Users\BrainInspired\AppData\Local\Temp\ipykernel_60840\2805368405.py:185: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.
  colormap = cm.get_cmap(cmap)
Creating GIF frames: 221it [00:10, 20.92it/s]


Saving GIF...
GIF saved to: ../output/cliphba_individual_visualization/dynamic_saliency_aardvark_01b.gif
